# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

C:\Program Files\Python39\python.exe


In [2]:
# Package installs are intentionally omitted; the class environment already includes these.
import importlib.util

for package in ["tensorflow", "tensorflow_model_optimization", "sklearn", "numpy", "pandas"]:
    status = "available" if importlib.util.find_spec(package) else "missing"
    print(f"{package}: {status}")

tensorflow: available
tensorflow_model_optimization: available
sklearn: available
numpy: available
pandas: available


In [3]:
import keras
print("Keras version:", keras.__version__)

Keras version: 2.14.0


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
to_categorical = tf.keras.utils.to_categorical

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

def file_size_kb(filename):
    return Path(filename).stat().st_size / 1024

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df.drop(columns=["Class"]).to_numpy(dtype=np.float32)
y = df["Class"].to_numpy(dtype=np.int64)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Label values:", np.unique(y))


X shape: (178, 13)
y shape: (178,)
Label values: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Training samples: 124
Test samples: 54


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

print("Scaled training feature mean (first 5):", np.round(X_train_scaled.mean(axis=0)[:5], 4))
print("Scaled training feature std (first 5):", np.round(X_train_scaled.std(axis=0)[:5], 4))

Scaled training feature mean (first 5): [ 0. -0.  0. -0. -0.]
Scaled training feature std (first 5): [1. 1. 1. 1. 1.]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)


y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential([
    Dense(64, activation="relu", input_shape=(num_features,)),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

model.summary()


Model: "sequential"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense (Dense)               (None, 64)                896       


 dense_1 (Dense)             (None, 32)                2080      


 dense_2 (Dense)             (None, 3)                 99        


Total params: 3075 (12.01 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

train_loss, train_accuracy = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
print(f"Final training accuracy: {train_accuracy:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

Final training accuracy: 0.9839
Final validation accuracy: 0.9200


In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
y_pred_base = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)

print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:\n", classification_report(y_test, y_pred_base))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_base))

Test accuracy: 0.9815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

converter = tf.lite.TFLiteConverter.from_keras_model(model)
model_base_tflite = converter.convert()
Path("model_base.tflite").write_bytes(model_base_tflite)

base_size_kb = file_size_kb("model_base.tflite")
print(f"Float32 TFLite model size: {base_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpcbkf63b2\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpcbkf63b2\assets


Float32 TFLite model size: 14.07 KB


## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    Path(filename).write_bytes(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_shape = input_details["shape"].copy()
    input_shape[0] = 1

    y_pred = []
    for sample in X_test.astype(np.float32):
        input_data = sample.reshape(input_shape)
        if input_details["dtype"] in (np.int8, np.uint8):
            scale, zero_point = input_details["quantization"]
            if scale > 0:
                input_data = np.round(input_data / scale + zero_point)
            dtype_info = np.iinfo(input_details["dtype"])
            input_data = np.clip(input_data, dtype_info.min, dtype_info.max).astype(input_details["dtype"])
        else:
            input_data = input_data.astype(input_details["dtype"])

        interpreter.set_tensor(input_details["index"], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details["index"])[0]
        if output_details["dtype"] in (np.int8, np.uint8):
            scale, zero_point = output_details["quantization"]
            if scale > 0:
                output_data = scale * (output_data.astype(np.float32) - zero_point)
        y_pred.append(int(np.argmax(output_data)))

    y_true = np.argmax(y_test_cat, axis=1)
    accuracy = accuracy_score(y_true, y_pred)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    print(f"{quant_type.upper()} TFLite accuracy: {accuracy:.4f}")
    print("Classification report:\n", classification_report(y_true, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

    return {"name": quant_type, "filename": filename, "size_kb": file_size_kb(filename), "accuracy": accuracy}


In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quant_results = {}
quant_results["int8"] = quantize_and_evaluate(model, X_test_scaled, y_test_cat, "int8", "model_int8.tflite")
quant_results["float16"] = quantize_and_evaluate(model, X_test_scaled, y_test_cat, "float16", "model_float16.tflite")
quant_results["dynamic"] = quantize_and_evaluate(model, X_test_scaled, y_test_cat, "dynamic", "model_dynamic.tflite")


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpmhpdm0nz\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpmhpdm0nz\assets


C:\Users\otaku\AppData\Roaming\Python\Python39\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 5.74 KB
INT8 TFLite accuracy: 0.9815
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpetevwy_2\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpetevwy_2\assets



FLOAT16 TFLite model size: 8.95 KB
FLOAT16 TFLite accuracy: 0.9815
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmp50_lgt6u\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmp50_lgt6u\assets



DYNAMIC TFLite model size: 8.17 KB
DYNAMIC TFLite accuracy: 0.9815
Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

prune_epochs = 10
prune_batch_size = 8
validation_fraction = 0.2
steps_per_epoch = int(np.ceil((len(X_train_scaled) * (1 - validation_fraction)) / prune_batch_size))
end_step = steps_per_epoch * prune_epochs

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.50,
    final_sparsity=0.70,
    begin_step=0,
    end_step=end_step,
)
pruning_params = {"pruning_schedule": pruning_schedule}

print("Pruning end_step:", end_step)

Pruning end_step: 130


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(Dense(64, activation="relu", input_shape=(num_features,)), **pruning_params),
    prune_low_magnitude(Dense(32, activation="relu"), **pruning_params),
    prune_low_magnitude(Dense(num_classes, activation="softmax"), **pruning_params),
])

pruned_model.summary()

Model: "sequential_1"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 prune_low_magnitude_dense_  (None, 64)                1730      


 3 (PruneLowMagnitude)                                           


...

 5 (PruneLowMagnitude)                                           


Total params: 6057 (23.67 KB)


Trainable params: 3075 (12.01 KB)


Non-trainable params: 2982 (11.66 KB)


_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_cat,
    epochs=prune_epochs,
    batch_size=prune_batch_size,
    validation_split=validation_fraction,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=0,
)

print(f"Final pruned validation accuracy: {pruned_history.history['val_accuracy'][-1]:.4f}")

Final pruned validation accuracy: 0.9200


In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
try:
    converter.optimizations = [tf.lite.Optimize.EXPERIMENTAL_SPARSITY]
except AttributeError:
    converter.optimizations = []
model_pruned_tflite = converter.convert()
Path("model_pruned.tflite").write_bytes(model_pruned_tflite)

pruned_size_kb = file_size_kb("model_pruned.tflite")
print(f"Pruned TFLite model size: {pruned_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpqvk9uiwf\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpqvk9uiwf\assets


Pruned TFLite model size: 8.06 KB


In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

pruned_probs = stripped_pruned_model.predict(X_test_scaled, verbose=0)
y_pred_pruned = np.argmax(pruned_probs, axis=1)
pruned_accuracy = accuracy_score(y_test, y_pred_pruned)

print(f"Pruned model test accuracy: {pruned_accuracy:.4f}")
print("Classification report:\n", classification_report(y_test, y_pred_pruned))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_pruned))

pruned_results = {"name": "pruned", "filename": "model_pruned.tflite", "size_kb": pruned_size_kb, "accuracy": pruned_accuracy}

Pruned model test accuracy: 0.9259
Classification report:
               precision    recall  f1-score   support

           0       0.90      1.00      0.95        18
           1       1.00      0.81      0.89        21
           2       0.88      1.00      0.94        15

    accuracy                           0.93        54
   macro avg       0.93      0.94      0.93        54
weighted avg       0.93      0.93      0.92        54

Confusion matrix:
 [[18  0  0]
 [ 2 17  2]
 [ 0  0 15]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation="relu", input_shape=(num_features,)),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

student_model.summary()

Model: "sequential_2"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 dense_6 (Dense)             (None, 32)                448       


 dense_7 (Dense)             (None, 16)                528       


 dense_8 (Dense)             (None, 3)                 51        


Total params: 1027 (4.01 KB)


Trainable params: 1027 (4.01 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled, verbose=0).astype(np.float32)

print("Teacher soft labels shape:", teacher_preds_soft.shape)
print("First teacher soft label:", np.round(teacher_preds_soft[0], 4))

Teacher soft labels shape: (124, 3)
First teacher soft label: [9.992e-01 4.000e-04 4.000e-04]


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

alpha = 0.5
y_train_combined = np.concatenate([y_train_cat.astype(np.float32), teacher_preds_soft], axis=1)

def hard_label_accuracy(y_true_combined, y_pred):
    return tf.keras.metrics.categorical_accuracy(y_true_combined[:, :num_classes], y_pred)

def distillation_loss(y_true_combined, y_pred):

    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    return alpha * hard_loss + (1 - alpha) * soft_loss

print("Combined distillation labels shape:", y_train_combined.shape)

Combined distillation labels shape: (124, 6)


In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer="adam",
    loss=distillation_loss,
    metrics=[hard_label_accuracy],
)

student_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

print(f"Final student validation accuracy: {student_history.history['val_hard_label_accuracy'][-1]:.4f}")

Final student validation accuracy: 0.9600


In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
model_kd_tflite = converter.convert()
Path("model_kd.tflite").write_bytes(model_kd_tflite)

kd_size_kb = file_size_kb("model_kd.tflite")
print(f"Knowledge distillation TFLite model size: {kd_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpgupo4_8i\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpgupo4_8i\assets


Knowledge distillation TFLite model size: 6.10 KB


In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

student_probs = student_model.predict(X_test_scaled, verbose=0)
y_pred_student = np.argmax(student_probs, axis=1)
kd_accuracy = accuracy_score(y_test, y_pred_student)

print(f"Knowledge distillation student test accuracy: {kd_accuracy:.4f}")
print("Classification report:\n", classification_report(y_test, y_pred_student))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_student))

kd_results = {"name": "kd", "filename": "model_kd.tflite", "size_kb": kd_size_kb, "accuracy": kd_accuracy}

Knowledge distillation student test accuracy: 0.9815
Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# Combine knowledge distillation with a smaller student and full int8 quantization.
previous_results = [
    {"name": "base_float32", "filename": "model_base.tflite", "size_kb": base_size_kb, "accuracy": test_accuracy},
    quant_results["int8"],
    quant_results["float16"],
    quant_results["dynamic"],
    pruned_results,
    kd_results,
]

previous_summary = pd.DataFrame(previous_results).sort_values("size_kb")
print("Previous model comparison:\n", previous_summary[["name", "size_kb", "accuracy"]].to_string(index=False))

tf.keras.backend.clear_session()
np.random.seed(SEED)
tf.random.set_seed(SEED)

part_e_alpha = 0.8

def compact_distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    return part_e_alpha * hard_loss + (1 - part_e_alpha) * soft_loss

tiny_student_model = Sequential([
    Dense(24, activation="relu", input_shape=(num_features,)),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax"),
])

tiny_student_model.compile(
    optimizer="adam",
    loss=compact_distillation_loss,
    metrics=[hard_label_accuracy],
)

tiny_history = tiny_student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=0,
)

print(f"Final compact student validation accuracy: {tiny_history.history['val_hard_label_accuracy'][-1]:.4f}")
tiny_kd_int8_results = quantize_and_evaluate(
    tiny_student_model,
    X_test_scaled,
    y_test_cat,
    "int8",
    "model_compact_kd_int8.tflite",
)
tiny_kd_int8_results["name"] = "compact_kd_int8"

all_results = previous_results + [tiny_kd_int8_results]
all_summary = pd.DataFrame(all_results).sort_values("size_kb")
print("\nAll model comparison:\n", all_summary[["name", "size_kb", "accuracy"]].to_string(index=False))

best_prior = min(previous_results, key=lambda result: result["size_kb"])
size_delta = best_prior["size_kb"] - tiny_kd_int8_results["size_kb"]
accuracy_delta = tiny_kd_int8_results["accuracy"] - best_prior["accuracy"]
print(
    f"\nPart (e) conclusion: {tiny_kd_int8_results['name']} is {size_delta:.2f} KB smaller than "
    f"the smallest prior model ({best_prior['name']}) with an accuracy change of {accuracy_delta:.4f}."
)

Previous model comparison:
         name   size_kb  accuracy
        int8  5.742188  0.981481
          kd  6.101562  0.981481
      pruned  8.062500  0.925926
     dynamic  8.171875  0.981481
     float16  8.945312  0.981481
base_float32 14.070312  0.981481


Final compact student validation accuracy: 0.9200


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpqzq4z7nd\assets


INFO:tensorflow:Assets written to: C:\Users\otaku\AppData\Local\Temp\tmpqzq4z7nd\assets


C:\Users\otaku\AppData\Roaming\Python\Python39\site-packages\tensorflow\lite\python\convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



INT8 TFLite model size: 3.34 KB
INT8 TFLite accuracy: 0.9815
Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       1.00      0.95      0.98        21
           2       1.00      1.00      1.00        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  0 15]]

All model comparison:
            name   size_kb  accuracy
compact_kd_int8  3.343750  0.981481
           int8  5.742188  0.981481
             kd  6.101562  0.981481
         pruned  8.062500  0.925926
        dynamic  8.171875  0.981481
        float16  8.945312  0.981481
   base_float32 14.070312  0.981481

Part (e) conclusion: compact_kd_int8 is 2.40 KB smaller than the smallest prior model (int8) with an accuracy change of 0.0000.


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
